# تست آزمایشی حرکت آوا — فقط GPU رایگان یا سیستم محلی

**اجرای این مدل در Arena انجام نشده است؛ نصب/استنتاج Colab هم هنوز آزمایش نشده‌اند.** این نوت‌بوک آماده‌سازی‌شده است، نه مدرک تولید حرکت. چهره و قاب آوا تأیید شده‌اند؛ دو تست گفتار و گوش‌دادن با EchoMimicV3 Flash، نه MuseTalk قبلی، اجرا می‌شوند.

1. از Runtime → Change runtime type، **GPU رایگان** را انتخاب کن. **اگر رایگان GPU نداد، پرداخت/ارتقا نده.**
2. سلول‌ها را به‌ترتیب اجرا کن. این یک اجرای کوتاه است اما وزن‌ها چندین گیگابایت‌اند؛ زمان/دسترسی GPU و سازگاری نصب تضمین نشده است.
3. در پایان ZIP نتیجه دانلود می‌شود؛ آن را برای بررسی و ترکیب در قاب اصلی برگردان.

عکس آوا و صدای مصنوعی زن06 در حساب Google خودت پردازش می‌شوند. هیچ صدای شخصی/کلونی، اتصال Drive، API پولی، اشتراک، رمز یا توکن لازم نیست. وزن‌های عمومی از منابع رسمی HF/ModelScope دانلود می‌شوند؛ عکس/صدا به آن دو سرویس آپلود نمی‌شوند. انتشار مجاز نیست.

گفتار کامل ۴٫۴۸ثانیه + سکوت انتهایی، بدون تغییر سرعت/گام؛ گوش‌دادن با ورودیِ سکوت و لب بسته، موسیقی فقط در پخش خروجی. حرکت سر/پلک/هماهنگی فارسی باید واقعاً روی فایل مدل بررسی شود. خروجی این مرحله هنوز فیلم نهایی نیست.

In [ ]:
# Stop BEFORE any large download if no suitable free GPU.
import sys, warnings
if sys.version_info[:2] not in ((3,10),(3,11)):
    warnings.warn('Python '+sys.version.split()[0]+' در فهرست محیط‌های تست‌شده سازنده نیست؛ علت خطای قبلی هنوز مشخص نیست.')
import torch
assert torch.cuda.is_available(), "GPU رایگان اختصاص داده نشده؛ متوقف شو و پرداخت نکن."
p = torch.cuda.get_device_properties(0)
assert p.total_memory >= 12 * 2**30, "حافظه GPU کم است؛ بدون خرید متوقف شو."
print(p.name, round(p.total_memory / 2**30, 2), "GiB VRAM")
print("از سطح رایگان استفاده کن؛ این کد دسترسی به وضعیت صورت‌حساب ندارد.")


In [ ]:
import pathlib, subprocess
ROOT = pathlib.Path('/content/ahangban-youtube')
BRANCH = 'arena/d4ba4e0b-ahangban-youtube'
def run(*args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
if not ROOT.exists():
    run('git', 'clone', '--filter=blob:none', '--no-checkout', '--depth=1', '--single-branch', '--branch', BRANCH, 'https://github.com/Masoumiofficial/ahangban-youtube.git', str(ROOT))
    run('git', 'sparse-checkout', 'set', 'content/011-ava-split-screen', cwd=ROOT)
    run('git', 'checkout', BRANCH, cwd=ROOT)
assert subprocess.check_output(['git','branch','--show-current'], cwd=ROOT, text=True).strip() == BRANCH
# Update only the SAME branch. Never reset local edits or delete cached results.
run('git', 'diff', '--quiet', cwd=ROOT)
run('git', 'diff', '--cached', '--quiet', cwd=ROOT)
run('git', 'fetch', 'origin', BRANCH, cwd=ROOT)
run('git', 'merge', '--ff-only', 'FETCH_HEAD', cwd=ROOT)
zip_path = ROOT / 'downloads/ahangban-ava-motion-pilot-inputs.zip'
zip_path.parent.mkdir(exist_ok=True)
with zip_path.open('wb') as out:
    run('git', 'show', 'HEAD:downloads/ahangban-ava-motion-pilot-inputs.zip', cwd=ROOT, stdout=out)
print('ورودی قفل‌شده بازیابی شد؛ صدا/چهره تازه تولید نشده است.')


In [ ]:
# Read-only: no model downloads, installation, deletion or upload.
# Safe separately; do NOT rerun inference to collect this diagnostic.
run(sys.executable, str(ROOT / 'content/011-ava-split-screen/motion/diagnose_runtime.py'))


In [ ]:
# Keep Colab's preinstalled matching CUDA Torch/torchvision. No paid services.
import sys
run(sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.49.0', 'accelerate==1.10.1',
    'huggingface-hub==0.34.4', 'modelscope==1.29.2', 'omegaconf==2.3.0',
    'einops==0.8.1', 'moviepy==2.2.1', 'librosa==0.11.0', 'pyloudnorm==0.1.1',
    'decord==0.6.0', 'timm==1.0.19', 'tomesd==0.1.3', 'torchdiffeq==0.2.5',
    'torchsde==0.2.6', 'ftfy==6.3.1', 'sentencepiece==0.2.1', 'imageio-ffmpeg==0.6.0',
    'opencv-python-headless==4.11.0.86', 'safetensors>=0.4.5')


In [ ]:
# Explicit inference only: this improves error reporting, NOT proof of successful model execution.
import sys, uuid
MOTION = ROOT / 'content/011-ava-split-screen/motion'
sys.path.insert(0, str(MOTION))
from process_tools import run_logged
log_path = pathlib.Path('/content/ava-motion-pilot/logs') / ('colab-run-' + uuid.uuid4().hex + '.log')
print('Local log:', log_path)
run_logged([sys.executable, '-u', str(MOTION / 'run_colab_pilot.py'), '--confirm-free-or-local'], log_path)


In [ ]:
from google.colab import files
result = pathlib.Path('/content/ava-motion-pilot/ava-motion-pilot-results.zip')
assert result.is_file(), 'نتیجه‌ای تولید نشده؛ فقط ورودی یا کد را نتیجهٔ حرکت حساب نکن.'
files.download(str(result))


## اگر خطا رخ داد

خروجی خطای سلول را برگردان؛ اگر GPU اختصاص داده نشد یا حافظه کافی نبود، ارتقا/خرید نکن. این تنظیمات کم‌حافظه با CPU-offload از قابلیت همان pipeline استفاده می‌کنند، اما در این محیط آزمایش سخت‌افزاری نشده‌اند و تضمین کارکرد روی T4 نیستند. وزن‌ مدل، صورت یا گفتار را برای رفع خطا به منابع غیررسمی منتقل نکن. هیچ توکن/رمز حسابی نفرست.

بعد از دریافت ZIP، هویت صورت/روسری، پلک، حرکت کوچک سر، کیفیت دندان/اتصال لب، هماهنگی فارسی، بسته‌ماندن لب در گوش‌دادن و ثبات میکروفون باید بررسی شوند. تغییر پیکسل یا موفقیت کد به‌تنهایی این کیفیت‌ها را ثابت نمی‌کند.
نسخهٔ جدید خروجی واقعی child را هم در سلول و هم در logs/colab-run-*.log ذخیره می‌کند؛ مرحلهٔ شکست در failure-summary.json ثبت می‌شود. فایل لاگِ قبلی پاک نمی‌شود. برای بررسی خطا، فقط سلول تشخیص را اجرا کن؛ Run all یا اجرای مجدد استنتاج لازم نیست. سلول دریافت کد، روی همان شاخه به‌روزرسانی fast-forward می‌کند و با تغییر محلی/تعارض متوقف می‌شود؛ reset یا حذف خروجی ندارد.

اصلاح بعدی: خطای Missing official model component: base/transformer از فرض اشتباه پوشه‌بندی بود؛ تنظیمات رسمی checkpointهای تخت در ریشه دارد. فیلتر و کنترل فایل‌ها اصلاح شد. برای ترمیم فقط وزن‌ها، بعد از به‌روزرسانی همان شاخه، runner را با --confirm-free-or-local --prepare-models-only اجرا کنید؛ این حالت هیچ مدل/ویدئویی تولید نمی‌کند. دانلود چندگیگابایتی ممکن است لازم باشد؛ کش قبلی حذف و دانلود اجباری نمی‌شود. رفع در Colab هنوز تأیید نشده است.
